# CHB-MIT Multi-Patient Preprocessing + Channel-Wise qEEG Feature Layer (dataset v1)

**Task scope.** Build and validate the FINAL data / preprocessing / feature layer for the
paper, BEFORE any final model is trained:

```
CHB-MIT (all eligible cases) -> generalized preprocessing (one implementation: chbmit_pipeline.py)
 -> montage compatibility rule -> 30 s non-overlapping epochs (never across recordings)
 -> seizure / event metadata -> CHANNEL-WISE qEEG features (final representation)
                              -> CHANNEL-AVERAGED qEEG features from the SAME epochs (ablation)
```

**This notebook does NOT** create binary labels for modeling, build LOSO folds, train
LR/RF/TCN, shuffle labels, compute SDI, or make paper figures. The validated labeling
(`EEG_binary_preictal_labeling.ipynb`) and LOSO (`EEG_loso_evaluation_splitting.ipynb`)
logic is not modified; `make_binary_labels` is copied verbatim below **only** to prove
backward compatibility and to report dataset counts.

**Signal processing is unchanged** from the validated CHB12/CHB24 notebooks (MNE FIR
band-pass 0.5-40 Hz, 30 s epochs, zero overlap, leftover samples dropped, Welch PSD with
2 s segments, the same 14 features). The only change is where spatial aggregation happens.

**Outputs** (new, versioned): `outputs/chbmit_multipatient_v1/`. The validated 2-patient
tables in `outputs/` are hashed at the start and re-checked unchanged at the end.

**Raw data location.** EDFs are cached outside the repo (C: had ~21 GB free; the corpus is
~43 GB), in the directory given by the `CHBMIT_RAW_DIR` environment variable (default
`data/raw/chbmit`). Every EDF is verified against PhysioNet's official `SHA256SUMS.txt`.

In [ ]:
import os, sys, json, hashlib, platform, time, re, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import mne, scipy

REPO = Path.cwd()
sys.path.insert(0, str(REPO))
import chbmit_pipeline as cp

RAW_DIR = Path(os.environ.get("CHBMIT_RAW_DIR", str(REPO / "data" / "raw" / "chbmit")))
META_DIR = RAW_DIR / "meta"
OUT = REPO / "outputs" / "chbmit_multipatient_v1"
CKPT = OUT / "checkpoints"
VAL = OUT / "validation"
for d in (OUT, CKPT, VAL):
    d.mkdir(parents=True, exist_ok=True)

# Development-only restriction (must be unset for the final recorded run; asserted in the report).
DEV_PATIENTS = [p for p in os.environ.get("CHBMIT_DEV_PATIENTS", "").split(",") if p]
WAIT_SEC = float(os.environ.get("CHBMIT_WAIT_SEC", "0"))

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "mne": mne.__version__, "scipy": scipy.__version__, "pyarrow": pa.__version__}
print("versions:", VERSIONS)
print("REPO   :", REPO)
print("RAW_DIR:", RAW_DIR)
print("OUT    :", OUT)
print("DEV_PATIENTS:", DEV_PATIENTS or "(none -- full run)")

def sha256(p):
    return cp.sha256_file(p)

# Validated legacy outputs that this task must NOT modify.
PROTECTED = sorted([p for p in (REPO / "outputs").glob("*.*")] +
                   [p for p in (REPO / "outputs" / "modeling").rglob("*") if p.is_file()])
PROTECTED_HASH_BEFORE = {str(p.relative_to(REPO)): sha256(p) for p in PROTECTED}
print(f"\nProtected legacy output files hashed: {len(PROTECTED_HASH_BEFORE)}")

## 1. Corpus metadata (official PhysioNet index + every case summary)

`RECORDS`, `RECORDS-WITH-SEIZURES`, `SHA256SUMS.txt`, `SUBJECT-INFO` and all 24
`chbXX-summary.txt` files are fetched if missing (AWS open-data mirror of PhysioNet,
physionet.org fallback). The repository's own `data/raw/chbmit/chb12|24-summary.txt`
(used by the validated notebooks) are byte-compared with the official copies.

In [ ]:
META_DIR.mkdir(parents=True, exist_ok=True)
def fetch_meta(rel, dest):
    if dest.exists():
        return "present"
    for base in (cp.S3_BASE_URL, cp.PHYSIONET_BASE_URL):
        try:
            urllib.request.urlretrieve(base + rel, dest)
            return "downloaded"
        except Exception as e:
            err = e
    raise RuntimeError(f"cannot fetch {rel}: {err}")

for f in ["RECORDS", "RECORDS-WITH-SEIZURES", "SHA256SUMS.txt", "SUBJECT-INFO"]:
    fetch_meta(f, META_DIR / f)
for i in range(1, 25):
    p = f"chb{i:02d}"
    fetch_meta(f"{p}/{p}-summary.txt", META_DIR / f"{p}-summary.txt")

index = cp.load_corpus_index(META_DIR)
n_sz_declared = sum(f["n_seizures"] for s in index["summaries"].values() for f in s["files"])
print(f"cases: {len(index['patients'])}  EDF records: {len(index['records'])}  "
      f"records-with-seizures: {len(index['records_with_seizures'])}  seizures in summaries: {n_sz_declared}")
for p, s in index["summaries"].items():
    assert s["sfreq"] in (None, 256.0), (p, s["sfreq"])

for p in ["chb12", "chb24"]:
    repo_copy = REPO / "data" / "raw" / "chbmit" / f"{p}-summary.txt"
    same = repo_copy.read_bytes() == (META_DIR / f"{p}-summary.txt").read_bytes()
    print(f"repo {p}-summary.txt identical to official copy: {same}")
    assert same

## 2. Recording table, annotation consistency, and patient timeline

Rules (explicit):
- **Seizure times** come from the case summary (the source used by the validated
  pipeline). A recording listed in `RECORDS-WITH-SEIZURES` but absent from its summary
  would be excluded (seizure times unknown).
- **Annotation conflicts** between the summary and `RECORDS-WITH-SEIZURES` are resolved with
  the official per-record `.seizures` annotation files (their presence in `SHA256SUMS.txt`).
- **Patient timeline**: file clock times (may exceed 24 h / wrap at midnight) are made
  monotonic by adding whole days. Gaps > 24 h are undetectable from clock times alone.

In [ ]:
legacy_ids = (set(pd.read_parquet(REPO / "outputs" / "features_chb24_forecasting.parquet")["recording_id"]) |
              set(pd.read_parquet(REPO / "outputs" / "features_chb12_forecasting.parquet")["recording_id"]))
sel = cp.build_selection(index, legacy_ids)
recs = sel["recordings"]
print(f"recordings: {len(recs)}")

conf = recs[recs["annotation_conflict"]].copy()
conf["official_.seizures_file_exists"] = conf.apply(
    lambda r: f"{r['patient_id']}/{r['edf_name']}.seizures" in index["sha256"], axis=1)
print("\nSummary vs RECORDS-WITH-SEIZURES conflicts:")
print(conf[["edf_name", "n_seizures", "in_records_with_seizures", "official_.seizures_file_exists"]].to_string(index=False))
# Resolution: the summary agrees with the official .seizures annotation files in every conflict.
assert ((conf["n_seizures"] > 0) == conf["official_.seizures_file_exists"]).all()
print("[OK] every conflict: summary agrees with the official .seizures annotation file -> summary used")

not_in_summary = recs[~recs["in_summary"]]
print(f"\nRecords absent from their summary: {len(not_in_summary)} -> {not_in_summary['edf_name'].tolist()}")
assert not not_in_summary["in_records_with_seizures"].any()
assert not any(f"{r.patient_id}/{r.edf_name}.seizures" in index["sha256"] for r in not_in_summary.itertuples())
print("[OK] none has seizures (not in RECORDS-WITH-SEIZURES, no .seizures file) -> seizure-free, "
      "but NO clock times -> cannot be placed on the patient timeline")

gaps = recs["gap_before_sec"].dropna()
print(f"\nInter-file gaps (s): median={gaps.median():.0f}, 90%={gaps.quantile(.9):.0f}, "
      f"max={gaps.max():.0f}; gaps > {cp.CONTIGUITY_GAP_TOLERANCE_SEC:.0f}s: {(gaps > cp.CONTIGUITY_GAP_TOLERANCE_SEC).sum()}")

## 3. Channel set / montage policy (explicit rule)

Start from the validated legacy 23-channel montage. Two derivations are redundant:
`T8-P8` appears twice (verified bitwise identical, re-checked per recording at runtime) and
`P7-T7` is the sign inversion of `T7-P7` (identical spectra). That leaves the 18 standard
longitudinal bipolar ("double banana") channels plus 3 FT derivations (`T7-FT9`,
`FT9-FT10`, `FT10-T8`).

**Rule (`cp.choose_channel_set`)**: keep the FT derivations only if every recording that
declares the 18 core channels also declares them; otherwise use the 18 core channels, so
no bipolar recording is lost for lack of an optional channel. Recordings lacking any
required channel are excluded (never interpolated / invented).

In [ ]:
CHANNELS = sel["channels"]
rule = sel["channel_rule"]
print(json.dumps({k: v for k, v in rule.items() if k != "channels"}, indent=1))
print(f"\nFINAL CHANNEL SET ({len(CHANNELS)}): {CHANNELS}")

decl = recs["declared_channels"].apply(json.loads)
ft_missing = recs[decl.apply(lambda d: cp.declared_has_channels(d, cp.DOUBLE_BANANA_18)) &
                  ~decl.apply(lambda d: cp.declared_has_channels(d, cp.DOUBLE_BANANA_18 + cp.FT_EXTENSION_3))]
print("\nFiles that declare the 18 core channels but lack an FT derivation (would be lost with 21 channels):")
print(ft_missing.groupby("patient_id").agg(files=("edf_name", "size"), seizures=("n_seizures", "sum")).to_string())

# Redundancy evidence on real data (one validated recording).
_edf = RAW_DIR / "chb24" / "chb24_01.edf"
_r = mne.io.read_raw_edf(_edf, preload=True, verbose="ERROR")
_d = _r.get_data(picks=["T8-P8-0", "T8-P8-1", "T7-P7", "P7-T7"])
print(f"\nchb24_01: T8-P8-0 == T8-P8-1 bitwise: {np.array_equal(_d[0], _d[1])}; "
      f"max|T7-P7 + P7-T7| = {np.abs(_d[2] + _d[3]).max():.2e} V vs max|T7-P7| = {np.abs(_d[2]).max():.2e} V "
      f"(quantization-level)")
del _r, _d

## 4. Patient inclusion rule

Defined before processing (constants in `chbmit_pipeline.py`):

| criterion | rule |
|---|---|
| seizure annotations | >= 1 annotated seizure in the summary |
| sufficient seizures for LOSO | >= `MIN_ELIGIBLE_SEIZURES` = 3 **lead** seizures with >= 1 PREICTAL epoch at H = 5 min (strictest horizon) that is >= 30 min after the end of every earlier annotated seizure of the case (patient timeline; within-recording where no timeline exists). With 3, every LOSO fold trains on >= 2 other seizures. |
| montage compatibility | counted only in recordings that declare all channels of the final set |
| usable interictal data | >= `MIN_INTERICTAL_HOURS` = 1 h of epochs >= 30 min from every annotated seizure |
| data integrity | every processed EDF must match the official SHA256, sfreq = 256 Hz, and pass runtime montage checks (Section 6) |

These are summary-level estimates on the 30 s epoch grid; they are re-computed on the
actual processed epochs in Section 9 and any change is reported.

In [ ]:
patients = sel["patients"]
plan = sel["plan"]
show = ["patient_id", "n_recordings", "n_recordings_compatible", "summary_hours_compatible",
        "n_seizures_annotated", "n_seizures_in_compatible", "eligible_seizures_within_est",
        "eligible_seizures_global_est", "interictal_h_global_est", "timeline_available", "included", "exclusion_reason"]
print(patients[show].round(1).to_string(index=False))
print(f"\nIncluded cases: {int(patients['included'].sum())} / {len(patients)}")
print("Note: chb21 is the same subject as chb01 recorded 1.5 years later (CHB-MIT documentation); "
      "kept as a separate case, flagged in the manifest.")

plan_path = OUT / "processing_plan.csv"
if plan_path.exists():
    saved = pd.read_csv(plan_path, keep_default_na=False)
    same = saved[plan.columns].astype(str).reset_index(drop=True).equals(plan.astype(str).reset_index(drop=True))
    print(f"\nprocessing_plan.csv (used by the unattended CLI run) identical to the re-derived plan: {same}")
    assert same
else:
    plan.to_csv(plan_path, index=False)
if DEV_PATIENTS:
    plan = plan[plan["patient_id"].isin(DEV_PATIENTS)].reset_index(drop=True)
print(f"plan rows: {len(plan)} ({plan['patient_id'].nunique()} cases); legacy-reproduction rows: "
      f"{(plan['legacy_channels'].fillna('') != '').sum()}")

## 5. Per-recording processing (idempotent; reuses checkpoints)

`cp.run_recording_to_checkpoint`: verified EDF -> keep the final channel set (plus, for the 17
validated legacy recordings only, the legacy 23 channels) -> MNE FIR 0.5-40 Hz -> 30 s
epochs inside the recording -> channel-wise + averaged features from the same epoch array.
The long first pass was run unattended with the same function through the module CLI
(`python chbmit_pipeline.py processing_plan.csv ...`); here every checkpoint is re-validated
(channel list, legacy flag, column schema) and reused, or recomputed if invalid.

In [ ]:
prov_rows, failures = [], []
t0 = time.time()
for i, r in plan.iterrows():
    cp.wait_for_file(RAW_DIR / r["patient_id"] / r["edf_name"], WAIT_SEC)
    legacy = json.loads(r["legacy_channels"]) if isinstance(r["legacy_channels"], str) and r["legacy_channels"] else None
    try:
        prov_rows.append(cp.run_recording_to_checkpoint(RAW_DIR, CKPT, r["patient_id"], r["edf_name"],
                                                        r["recording_id"], r["sha256_expected"],
                                                        json.loads(r["channels"]), legacy))
    except Exception as e:
        failures.append({"patient_id": r["patient_id"], "edf_name": r["edf_name"],
                         "recording_id": r["recording_id"], "error": f"{type(e).__name__}: {e}"})
prov = pd.DataFrame(prov_rows)
print(f"processed/reused: {len(prov)}  failed: {len(failures)}  ({time.time()-t0:.0f}s)")
print(prov["checkpoint_status"].value_counts().to_string())
if failures:
    print(pd.DataFrame(failures).to_string(index=False))

## 6. Runtime verification (sfreq, montage, duplicates, durations, integrity)

In [ ]:
recs_i = recs.set_index("recording_id")
prov["summary_duration_sec"] = prov["recording_id"].map(recs_i["summary_duration_sec"])
prov["duration_diff_vs_summary_sec"] = prov["duration_sec"] - prov["summary_duration_sec"]
prov["expected_n_epochs"] = (prov["n_times"] // int(cp.EPOCH_SEC * cp.EXPECTED_SFREQ)).astype(int)

assert (prov["sfreq"] == cp.EXPECTED_SFREQ).all()
assert (prov["n_epochs"] == prov["expected_n_epochs"]).all()
assert prov["channels"].apply(lambda c: c == CHANNELS).all()
assert (prov["sha256"] == prov["recording_id"].map(recs_i["sha256_expected"])).all()
dup = prov["duplicate_labels_identical"].apply(lambda d: all(d.values()) if d else True)  # data property, see Section 11
print(f"sfreq 256 Hz: all {len(prov)}; n_epochs == floor(n_times/7680): all; channel list == final set: all")
print(f"recordings where a used channel label is duplicated: "
      f"{(prov['duplicate_labels_identical'].apply(len) > 0).sum()}; duplicates bitwise identical: {dup.all()} "
      f"(non-identical: {prov.loc[~dup, 'recording_id'].tolist()})")
dd = prov["duration_diff_vs_summary_sec"].dropna()
print(f"EDF duration - summary duration (s): min={dd.min():.0f} median={dd.median():.0f} max={dd.max():.0f}; "
      f"|diff| > 5 s: {(dd.abs() > 5).sum()}")
print(prov.loc[prov["duration_diff_vs_summary_sec"].abs() > 5,
               ["recording_id", "duration_sec", "summary_duration_sec", "duration_diff_vs_summary_sec"]].to_string(index=False))
print(f"leftover seconds dropped per recording: max={prov['leftover_sec'].max():.2f}")
print(f"download status: {prov['download_status'].value_counts().to_dict()}")

# channel/montage manifest (long form): which runtime channel carried each final channel
chan_manifest = pd.DataFrame([
    {"patient_id": r.patient_id, "recording_id": r.recording_id, "channel_order": k, "channel": ch,
     "channel_normalized": cp.normalize_channel_name(ch), "runtime_channel": r.channel_map[ch],
     "n_runtime_channels_in_edf": len(r.runtime_channels),
     "runtime_channels_dropped": json.dumps([c for c in r.runtime_channels if c not in r.channel_map.values()])}
    for r in prov.itertuples() for k, ch in enumerate(CHANNELS)])
renamed = chan_manifest[chan_manifest["channel"] != chan_manifest["runtime_channel"]]
print(f"\nfinal channels resolved via an MNE running-number duplicate name: "
      f"{renamed.groupby(['channel', 'runtime_channel']).size().to_dict()}")

## 7. Seizure events and per-epoch metadata

Within-recording columns reproduce the validated definitions exactly
(`y_seizure`, `seizure_id`, `next_seizure_*`, `time_to_next_seizure_sec`,
`previous_seizure_*`, `time_since_previous_seizure_sec`). **Additional, descriptive** columns
use the patient timeline and every annotated seizure of the case (including seizures in
recordings that were not processed): `global_*`, `abs_t_*`, and `observed_before_sec` /
`observed_after_sec` (continuous recorded time around the epoch; consecutive files with gaps
<= 60 s are one block). They are provided so the labeling task can decide the
cross-recording rule; nothing here changes the validated labels.

In [ ]:
seizures_all = pd.concat([cp.patient_seizure_table(g) for _, g in recs.groupby("patient_id")], ignore_index=True)
processed_ids = set(prov["recording_id"])
seizures_all["recording_processed"] = seizures_all["recording_id"].isin(processed_ids)
seizures_all["patient_included"] = seizures_all["patient_id"].isin(plan["patient_id"].unique())
print(f"annotated seizures (all cases): {len(seizures_all)}; in processed recordings: "
      f"{int(seizures_all['recording_processed'].sum())}")

meta_parts = []
for p, g in prov.groupby("patient_id", sort=False):
    prec = recs[recs["patient_id"] == p]
    blocks = cp.contiguous_blocks(prec)
    psz = seizures_all[seizures_all["patient_id"] == p]
    for r in g.itertuples():
        rr = recs_i.loc[r.recording_id].to_dict()
        rr.update({"recording_id": r.recording_id, "duration_sec": r.duration_sec})
        meta_parts.append(cp.epoch_metadata_for_recording(rr, r.n_epochs, psz, blocks.get(r.recording_id)))
epoch_meta = pd.concat(meta_parts, ignore_index=True)
epoch_meta.insert(0, "epoch_uid", epoch_meta["recording_id"] + "_e" + epoch_meta["epoch_idx"].astype(str).str.zfill(5))
epoch_meta["recording_is_seizure_free"] = epoch_meta["recording_n_seizures"] == 0
print(f"epoch metadata: {epoch_meta.shape}")

## 8. Feature tables (streamed from checkpoints, same epochs, same row order)

- `features_channelwise.parquet` -- FINAL representation: `<feature>__<CHANNEL>` for each of
  the 18 channels x 14 features = 252 columns, channel-major / feature-minor order.
- `features_channel_averaged.parquet` -- ABLATION: the 14 legacy-named columns (`std`, ...,
  `entropy`) computed with the legacy averaging formulas over the SAME 18 channels and the
  SAME epoch arrays (only spatial aggregation differs).
- Legacy 23-channel averages (validated recordings only) stay in the checkpoints and are
  used solely for the backward-reproduction check.

In [ ]:
CW_COLS = cp.channelwise_columns(CHANNELS)
KEY = ["patient_id", "recording_id", "epoch_idx"]
cw_path, av_path = OUT / "features_channelwise.parquet", OUT / "features_channel_averaged.parquet"
w_cw = w_av = None
n_rows = n_nonfinite_cw = n_nonfinite_av = 0
legacy_parts = []
for r in prov.itertuples():
    f = pd.read_parquet(cp.checkpoint_paths(CKPT, r.patient_id, r.recording_id)[0])
    assert f["epoch_idx"].tolist() == list(range(r.n_epochs))
    f.insert(0, "patient_id", r.patient_id)
    cw = f[KEY + CW_COLS]
    av = f[KEY + [f"avg__{x}" for x in cp.FEATURES]].rename(columns={f"avg__{x}": x for x in cp.FEATURES})
    n_nonfinite_cw += int((~np.isfinite(cw[CW_COLS].to_numpy())).sum())
    n_nonfinite_av += int((~np.isfinite(av[cp.FEATURES].to_numpy())).sum())
    t_cw, t_av = pa.Table.from_pandas(cw, preserve_index=False), pa.Table.from_pandas(av, preserve_index=False)
    if w_cw is None:
        w_cw, w_av = pq.ParquetWriter(cw_path, t_cw.schema), pq.ParquetWriter(av_path, t_av.schema)
    w_cw.write_table(t_cw); w_av.write_table(t_av)
    n_rows += len(f)
    if "legacy23__std" in f.columns:
        legacy_parts.append(f[KEY + [f"legacy23__{x}" for x in cp.FEATURES]])
w_cw.close(); w_av.close()
legacy23 = pd.concat(legacy_parts, ignore_index=True)
print(f"rows written: {n_rows}; channel-wise columns: {len(CW_COLS)}; averaged columns: {len(cp.FEATURES)}")
print(f"non-finite values: channel-wise={n_nonfinite_cw}, averaged={n_nonfinite_av}")
assert n_rows == len(epoch_meta)

## 9. Backward validation against the validated CHB12 / CHB24 tables

Compared: recordings, epoch counts, timestamps, seizure intervals (verbatim legacy
`parse_chb_summary`), per-epoch seizure metadata, feature values (legacy 23-channel average
vs the saved 14 features), and labels/exclusions (verbatim `make_binary_labels` applied to
the new metadata vs `labels_binary_preictal.parquet`, all three horizons).

In [ ]:
# --- verbatim copies from the validated notebooks (EEG_CHB24_preprocessing.ipynb /
# --- EEG_binary_preictal_labeling.ipynb); do not edit independently.
def parse_chb_summary(summary_path, known_edf_filenames):
    text = summary_path.read_text()
    blocks = re.split(r"(?=File Name: )", text)
    annotations = {}
    for block in blocks:
        name_match = re.search(r"File Name:\s*(\S+)", block)
        if not name_match:
            continue
        fname = name_match.group(1)
        if fname not in known_edf_filenames:
            continue
        n_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        if not n_match:
            raise ValueError(f"{fname}: could not find seizure count in summary")
        n_seizures = int(n_match.group(1))
        starts = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds", block)]
        ends = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds", block)]
        if len(starts) != n_seizures or len(ends) != n_seizures:
            raise ValueError(f"{fname}: declared {n_seizures} seizures but parsed {len(starts)} start / {len(ends)} end times")
        annotations[fname] = list(zip(starts, ends))
    missing = set(known_edf_filenames) - set(annotations)
    if missing:
        raise ValueError(f"No summary entry found for: {missing}")
    return annotations

def make_binary_labels(df, preictal_minutes=5, exclusion_minutes=30):
    if "time_since_previous_seizure_sec" not in df.columns:
        raise ValueError("Run add_seizure_adjacency_metadata() first.")
    if exclusion_minutes < 30:
        raise ValueError("Supervisor requirement: exclusion_minutes must be >= 30.")
    if preictal_minutes > exclusion_minutes:
        raise ValueError("preictal_minutes must be <= exclusion_minutes.")
    preictal_sec = preictal_minutes * 60.0
    exclusion_sec = exclusion_minutes * 60.0
    y_seizure = df["y_seizure"].to_numpy().astype(bool)
    dt_next = df["time_to_next_seizure_sec"].to_numpy()
    dt_prev = df["time_since_previous_seizure_sec"].to_numpy()
    is_near_future = (~np.isnan(dt_next)) & (dt_next <= exclusion_sec)
    is_near_past = (~np.isnan(dt_prev)) & (dt_prev <= exclusion_sec)
    is_preictal_horizon = (~np.isnan(dt_next)) & (dt_next > 0) & (dt_next <= preictal_sec)
    preictal_mask = (~y_seizure) & is_preictal_horizon & (~is_near_past)
    interictal_mask = (~y_seizure) & (~is_near_future) & (~is_near_past)
    label = np.full(len(df), None, dtype=object)
    reason = np.full(len(df), None, dtype=object)
    label[preictal_mask] = "PREICTAL"
    label[interictal_mask] = "INTERICTAL"
    excluded = ~(preictal_mask | interictal_mask)
    reason[excluded & y_seizure] = "seizure_overlap"
    remaining = excluded & (~y_seizure) & (reason == None)
    reason[remaining & is_near_past] = "postictal_exclusion"
    remaining = excluded & (~y_seizure) & (reason == None)
    ambiguous_gap = remaining & (~np.isnan(dt_next)) & (dt_next > preictal_sec) & (dt_next <= exclusion_sec)
    reason[ambiguous_gap] = "ambiguous_preictal_gap"
    remaining = excluded & (~y_seizure) & (reason == None)
    onset_boundary = remaining & (~np.isnan(dt_next)) & (dt_next == 0)
    reason[onset_boundary] = "seizure_onset_boundary"
    remaining = excluded & (~y_seizure) & (reason == None)
    if remaining.any():
        raise AssertionError(f"{remaining.sum()} epochs fell through labeling logic unexplained")
    return label, reason

HORIZONS = [5, 15, 30]

In [ ]:
bv = {}
old = pd.concat([pd.read_parquet(REPO / "outputs" / "features_chb24_forecasting.parquet"),
                 pd.read_parquet(REPO / "outputs" / "features_chb12_forecasting.parquet")], ignore_index=True)
labels_old = pd.read_parquet(REPO / "outputs" / "labels_binary_preictal.parquet")
new = epoch_meta[epoch_meta["recording_id"].isin(set(old["recording_id"]))].merge(
    legacy23, on=["patient_id", "recording_id", "epoch_idx"], how="left", validate="1:1")

# recordings + epoch counts
oc, nc = old.groupby("recording_id").size(), new.groupby("recording_id").size()
bv["recordings_old"] = int(len(oc)); bv["recordings_matched"] = int(oc.index.isin(nc.index).sum())
bv["epoch_counts_equal"] = bool(oc.equals(nc.reindex(oc.index)))
print(f"recordings: old {len(oc)}, found in new {bv['recordings_matched']}; epoch counts identical: {bv['epoch_counts_equal']}")

m = old.rename(columns={f: f"{f}_old" for f in cp.FEATURES}).merge(
    new, on=["recording_id", "epoch_idx"], suffixes=("_old", ""), validate="1:1")
assert len(m) == len(old)
for c in ["t_start_sec", "t_center_sec", "t_end_sec", "y_seizure"]:
    bv[f"{c}_equal"] = bool((m[f"{c}_old"] == m[c]).all())
for c in ["next_seizure_start_sec", "time_to_next_seizure_sec"]:
    bv[f"{c}_equal"] = bool(np.array_equal(m[f"{c}_old"].to_numpy(float), m[c].to_numpy(float), equal_nan=True))
print({k: v for k, v in bv.items() if k.endswith("_equal")})

# seizure intervals: verbatim legacy parser on the repo summary files vs new seizure events
iv_ok = True
for p in ["chb12", "chb24"]:
    rids = sorted(set(old.loc[old["recording_id"].str.startswith(f"chb_s{p[3:]}"), "recording_id"]))
    fn = {rid: f"{p}_{rid.split('_r')[1]}.edf" for rid in rids}
    legacy_iv = parse_chb_summary(REPO / "data" / "raw" / "chbmit" / f"{p}-summary.txt", set(fn.values()))
    for rid, f in fn.items():
        ne = seizures_all[seizures_all["recording_id"] == rid].sort_values("seizure_index_in_file")
        iv_ok &= [tuple(map(int, x)) for x in ne[["onset_sec", "offset_sec"]].to_numpy()] == [tuple(x) for x in legacy_iv[f]]
bv["seizure_intervals_equal"] = bool(iv_ok)
print(f"seizure intervals identical to the verbatim legacy parser: {iv_ok}")

# adjacency metadata vs the validated labels table
ml = labels_old.merge(new, on=["recording_id", "epoch_idx"], suffixes=("_old", ""), validate="1:1")
for c in ["seizure_id", "next_seizure_id", "previous_seizure_id"]:
    bv[f"{c}_equal"] = bool((ml[f"{c}_old"].astype(object).where(ml[f"{c}_old"].notna(), None).tolist() ==
                             ml[c].astype(object).where(ml[c].notna(), None).tolist()))
for c in ["previous_seizure_end_sec", "time_since_previous_seizure_sec"]:
    bv[f"{c}_equal"] = bool(np.array_equal(ml[f"{c}_old"].to_numpy(float), ml[c].to_numpy(float), equal_nan=True))

# labels / exclusions (verbatim make_binary_labels on the NEW metadata)
lab_new = new.set_index(["recording_id", "epoch_idx"]).sort_index()
lab_old = labels_old.set_index(["recording_id", "epoch_idx"]).sort_index()
for H in HORIZONS:
    lab, rea = make_binary_labels(lab_new, H, 30)
    assert lab_new.index.equals(lab_old.index)
    # compare values as plain Python lists (pandas 3 infers different dtypes for None/NaN-filled columns)
    to_list = lambda a: [None if (v is None or (isinstance(v, float) and np.isnan(v))) else str(v) for v in a]
    bv[f"labels_h{H}_equal"] = to_list(lab) == to_list(lab_old[f"binary_label_h{H}"].astype(object))
    bv[f"exclusion_reasons_h{H}_equal"] = to_list(rea) == to_list(lab_old[f"exclusion_reason_h{H}"].astype(object))
    print(f"H={H:2d}: new labels {pd.Series(to_list(lab)).value_counts(dropna=False).to_dict()}")
print(json.dumps(bv, indent=1))

# feature values: legacy 23-channel average vs saved validated features
diff_rows = []
for f in cp.FEATURES:
    a, b = m[f"{f}_old"].to_numpy(float), m[f"legacy23__{f}"].to_numpy(float)
    rel = np.abs(a - b) / np.maximum(np.abs(a), 1e-300)
    diff_rows.append({"feature": f, "max_abs_diff": float(np.abs(a - b).max()), "max_rel_diff": float(rel.max()),
                      "allclose_rtol1e-9": bool(np.allclose(a, b, rtol=1e-9, atol=0))})
feat_diff = pd.DataFrame(diff_rows)
print(feat_diff.to_string(index=False))
bv["features_legacy23_allclose_rtol1e-9"] = bool(feat_diff["allclose_rtol1e-9"].all())
bv["features_legacy23_max_rel_diff"] = float(feat_diff["max_rel_diff"].max())

# expected, by-design differences: final 18-channel average vs legacy 23-channel average
av_new = pd.read_parquet(av_path, filters=[("recording_id", "in", sorted(set(old["recording_id"])))])
d18 = av_new.merge(legacy23, on=KEY)
design = pd.DataFrame([{"feature": f, "median_rel_diff_18_vs_23": float(np.median(np.abs(d18[f] - d18[f"legacy23__{f}"]) /
                        np.abs(d18[f"legacy23__{f}"]))), "max_rel_diff_18_vs_23": float(np.max(np.abs(d18[f] - d18[f"legacy23__{f}"]) /
                        np.abs(d18[f"legacy23__{f}"])))} for f in cp.FEATURES])
print("\nBy-design difference (final 18-channel average vs legacy 23-channel average, same epochs):")
print(design.round(4).to_string(index=False))
feat_diff.to_csv(VAL / "backward_feature_diffs.csv", index=False)
design.to_csv(VAL / "averaged18_vs_legacy23_design_difference.csv", index=False)

chb12_new = sorted(epoch_meta.loc[epoch_meta["patient_id"] == "chb12", "recording_id"].unique())
bv["chb12_recordings_new"] = len(chb12_new)
bv["chb12_recordings_old"] = int(old["recording_id"].str.startswith("chb_s12").groupby(old["recording_id"]).first().sum())
bv["all_checks_passed"] = bool(all(v for k, v in bv.items() if k.endswith("_equal") or k.startswith("features_legacy23_allclose"))
                               and bv["recordings_matched"] == bv["recordings_old"])
print(f"\nBACKWARD VALIDATION PASSED: {bv['all_checks_passed']}")

## 10. Dataset counts with the validated labeling rule (descriptive only)

`make_binary_labels` (verbatim, within-recording distances = the validated rule) is applied
to every processed epoch to report counts and to re-check the inclusion rule on real epochs.
A second, **descriptive** variant substitutes the patient-timeline distances
(`global_*`) to quantify how many labels the within-recording blind spot affects. No label
table for modeling is written here.

In [ ]:
lab_rows, glob_rows = {}, {}
for H in HORIZONS:
    lab_rows[H], _ = make_binary_labels(epoch_meta, H, 30)
    gdf = epoch_meta[["y_seizure"]].copy()
    gdf["time_to_next_seizure_sec"] = epoch_meta["global_time_to_next_seizure_sec"].where(
        epoch_meta["timeline_available"], epoch_meta["time_to_next_seizure_sec"])
    gdf["time_since_previous_seizure_sec"] = epoch_meta["global_time_since_previous_seizure_sec"].where(
        epoch_meta["timeline_available"], epoch_meta["time_since_previous_seizure_sec"])
    glob_rows[H], _ = make_binary_labels(gdf, H, 30)

counts = []
for p, g in epoch_meta.groupby("patient_id", sort=False):
    pr = prov[prov["patient_id"] == p]
    row = {"patient_id": p, "recordings": len(pr), "seizure_free_recordings": int((pr["recording_id"].map(
               g.groupby("recording_id")["recording_n_seizures"].first()) == 0).sum()),
           "hours": pr["duration_sec"].sum() / 3600, "epochs": len(g),
           "seizures_annotated_in_processed": int(seizures_all[(seizures_all["patient_id"] == p) & seizures_all["recording_processed"]].shape[0]),
           "seizure_epochs": int(g["y_seizure"].sum())}
    for H in HORIZONS:
        l = pd.Series(lab_rows[H][g.index], index=g.index)
        lg = pd.Series(glob_rows[H][g.index], index=g.index)
        row[f"preictal_h{H}"] = int((l == "PREICTAL").sum())
        row[f"eligible_seizures_h{H}"] = int(g.loc[l == "PREICTAL", "next_seizure_id"].nunique())
        gnext = g["global_next_seizure_id"].where(g["timeline_available"], g["next_seizure_id"])
        row[f"eligible_seizures_global_h{H}"] = int(gnext[lg == "PREICTAL"].nunique())
        if H == 5:
            row["interictal_epochs"] = int((l == "INTERICTAL").sum())
            row["interictal_h"] = row["interictal_epochs"] * cp.EPOCH_SEC / 3600
            row["interictal_in_seizure_free_recs_h"] = int(((l == "INTERICTAL") & g["recording_is_seizure_free"]).sum()) * cp.EPOCH_SEC / 3600
            row["interictal_epochs_global"] = int((lg == "INTERICTAL").sum())
            row["within_vs_global_label_changes_h5"] = int((l.fillna("X") != lg.fillna("X")).sum())
    counts.append(row)
dataset_counts = pd.DataFrame(counts)
dataset_counts["meets_rule_after_processing"] = ((dataset_counts["eligible_seizures_global_h5"] >= cp.MIN_ELIGIBLE_SEIZURES) &
                                                 (dataset_counts["interictal_h"] >= cp.MIN_INTERICTAL_HOURS))
pd.set_option("display.width", 250)
print(dataset_counts.round(2).to_string(index=False))
tot = dataset_counts.drop(columns="patient_id").sum(numeric_only=True)
print(f"\nTOTAL: {len(dataset_counts)} cases, {int(tot['recordings'])} recordings, {tot['hours']:.1f} h, "
      f"{int(tot['epochs'])} epochs, {int(tot['seizures_annotated_in_processed'])} seizures in processed recordings, "
      f"eligible lead seizures H=5 (validated rule) {int(tot['eligible_seizures_h5'])}, "
      f"(timeline rule) {int(tot['eligible_seizures_global_h5'])}, interictal {tot['interictal_h']:.1f} h")
print("cases failing the inclusion rule on real epochs:",
      dataset_counts.loc[~dataset_counts["meets_rule_after_processing"], "patient_id"].tolist())

## 11. Quality checks

In [ ]:
qc = []
def check(name, passed, detail=""):
    qc.append({"check": name, "passed": bool(passed), "detail": str(detail)})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")

check("no NaN/Inf in channel-wise features", n_nonfinite_cw == 0, f"{n_nonfinite_cw} non-finite values")
check("no NaN/Inf in channel-averaged features", n_nonfinite_av == 0, f"{n_nonfinite_av} non-finite values")
cw_schema = pq.read_schema(cw_path).names
check("deterministic channel-wise columns", cw_schema == KEY + CW_COLS,
      f"{len(CW_COLS)} cols = {len(CHANNELS)} channels x {len(cp.FEATURES)} features, channel-major order")
check("consistent channel set for every recording", prov["channels"].apply(lambda c: c == CHANNELS).all(),
      f"{len(prov)} recordings x {len(CHANNELS)} channels")
check("epoch length 30 s", ((epoch_meta["t_end_sec"] - epoch_meta["t_start_sec"]) == cp.EPOCH_SEC).all(), "t_end - t_start")
check("sampling frequency 256 Hz (7680 samples/epoch)", (prov["sfreq"] == 256.0).all(), "all processed EDFs")
dur = epoch_meta["recording_id"].map(prov.set_index("recording_id")["duration_sec"])
contig = epoch_meta.groupby("recording_id")["epoch_idx"].apply(lambda s: s.tolist() == list(range(len(s)))).all()
check("no epoch crosses a recording boundary", (epoch_meta["t_end_sec"] <= dur + 1e-9).all() and contig,
      "t_end <= recording duration; epoch_idx contiguous 0..n-1 per recording")

# independent recomputation of seizure-overlap and time-to-next metadata (brute force)
yb = np.zeros(len(epoch_meta), int); ttn = np.full(len(epoch_meta), np.nan)
szp = seizures_all[seizures_all["recording_processed"]].groupby("recording_id")
for rid, idx_ in epoch_meta.groupby("recording_id").groups.items():
    if rid not in szp.groups:
        continue
    s = szp.get_group(rid)
    ts, te = epoch_meta.loc[idx_, "t_start_sec"].to_numpy(), epoch_meta.loc[idx_, "t_end_sec"].to_numpy()
    on, off = s["onset_sec"].to_numpy(), s["offset_sec"].to_numpy()
    yb[idx_] = ((ts[:, None] < off[None]) & (te[:, None] > on[None])).any(1).astype(int)
    d = on[None, :] - te[:, None]
    d = np.where(d >= 0, d, np.inf).min(1)
    ttn[idx_] = np.where(np.isfinite(d), d, np.nan)
check("seizure-overlap metadata correct", (yb == epoch_meta["y_seizure"].to_numpy()).all(), "independent interval-intersection recomputation")
check("time-to-next-seizure metadata correct",
      np.array_equal(ttn, epoch_meta["time_to_next_seizure_sec"].to_numpy(float), equal_nan=True)
      and (epoch_meta["time_to_next_seizure_sec"].dropna() >= 0).all(), "independent brute-force recomputation, non-negative")
sz_proc = seizures_all[seizures_all["recording_processed"]]
covered = sz_proc["seizure_id"].isin(set(epoch_meta.loc[epoch_meta["y_seizure"] == 1, "seizure_id"]) |
                                     set(epoch_meta["seizure_id"].dropna()))
check("every annotated seizure in a processed recording overlaps >= 1 epoch", covered.all(),
      f"{int(covered.sum())}/{len(sz_proc)}; not covered: {sz_proc.loc[~covered, 'seizure_id'].tolist()}")
check("patient identity preserved", (epoch_meta["recording_id"].str[5:7].astype(int) == epoch_meta["subject_id"]).all()
      and (epoch_meta["patient_id"].str[3:].astype(int) == epoch_meta["subject_id"]).all()
      and epoch_meta.groupby("recording_id")["patient_id"].nunique().max() == 1, "recording_id <-> patient_id <-> subject_id")
check("recording identity preserved", prov["recording_id"].is_unique and prov["edf_name"].is_unique
      and (epoch_meta.groupby("recording_id")["edf_name"].nunique() == 1).all(), "recording_id <-> edf_name 1:1")
check("no duplicate epochs", not epoch_meta.duplicated(["recording_id", "epoch_idx"]).any()
      and epoch_meta["epoch_uid"].is_unique, f"{len(epoch_meta)} unique (recording_id, epoch_idx)")
check("SHA256 integrity of every processed EDF", (prov["sha256"] == prov["recording_id"].map(recs_i["sha256_expected"])).all()
      and prov["download_status"].isin(["downloaded", "verified_existing"]).all(), "official SHA256SUMS.txt")
# Duplicate EDF labels (MNE running numbers) of a used channel: rule = use the FIRST occurrence
# (standard double-banana chain position). Bitwise identity is a data property, reported as a finding.
dup_findings = []
for r in prov.itertuples():
    for want, same in r.duplicate_labels_identical.items():
        dups = [n for n in r.runtime_channels if re.fullmatch(re.escape(want) + r"-\d+", n)]
        row = {"recording_id": r.recording_id, "channel": want, "labels": dups, "bitwise_identical": bool(same),
               "used_runtime_channel": r.channel_map[want], "first_occurrence_used": r.channel_map[want] == dups[0]}
        if not same:
            raw_ = mne.io.read_raw_edf(RAW_DIR / r.patient_id / r.edf_name, preload=False, verbose="ERROR")
            raw_.pick(dups).load_data(verbose="ERROR")
            row["corr_between_duplicates"] = float(np.corrcoef(raw_.get_data())[0, 1])
            del raw_
        dup_findings.append(row)
dup_findings = pd.DataFrame(dup_findings)
dup_findings.to_csv(VAL / "duplicate_channel_labels.csv", index=False)
nonid = dup_findings[~dup_findings["bitwise_identical"]]
print(f"FINDING: duplicated label of a used channel in {len(dup_findings)} recordings; bitwise identical in "
      f"{int(dup_findings['bitwise_identical'].sum())}; NOT identical in {len(nonid)}: "
      f"{dict(zip(nonid['recording_id'], nonid['corr_between_duplicates'].round(3)))} (correlation between the two)")
check("duplicate labels: first-occurrence rule applied and non-identical cases documented",
      dup_findings["first_occurrence_used"].all() and nonid["corr_between_duplicates"].notna().all(),
      f"{len(nonid)} non-identical duplicates documented in validation/duplicate_channel_labels.csv")
included = set(plan["patient_id"])
rec_status = recs[["patient_id", "edf_name", "recording_id", "exclusion_reason_pre"]].copy()
rec_status["status"] = np.where(rec_status["recording_id"].isin(processed_ids), "processed",
                        np.where(rec_status["exclusion_reason_pre"] != "", "excluded_before_processing",
                        np.where(~rec_status["patient_id"].isin(included), "patient_not_included",
                        np.where(rec_status["recording_id"].isin([f["recording_id"] for f in failures]), "failed", "UNACCOUNTED"))))
check("all recordings accounted for / exclusions documented", (rec_status["status"] != "UNACCOUNTED").all()
      and (rec_status.loc[rec_status["status"] == "excluded_before_processing", "exclusion_reason_pre"] != "").all(),
      rec_status["status"].value_counts().to_dict())
check("no processing failures", len(failures) == 0, f"{len(failures)} failures")

# descriptive signal-quality flag (NOT rejection): flat channels per epoch
std_cols = [f"std__{cp.normalize_channel_name(c)}" for c in CHANNELS]
std_tab = pd.read_parquet(cw_path, columns=KEY + std_cols)
FLAT_STD_V = 1e-7  # 0.1 uV
n_flat = (std_tab[std_cols].to_numpy() < FLAT_STD_V).sum(1)
epoch_meta["n_flat_channels"] = n_flat
print(f"\nDescriptive only: epochs with >= 1 flat channel (std < 0.1 uV): {(n_flat > 0).sum()} "
      f"({(n_flat > 0).mean():.2%}); all 18 flat: {(n_flat == len(CHANNELS)).sum()}")
print("ARTIFACT REJECTION: NOT IMPLEMENTED. No epoch is removed or modified for artifacts; "
      "`n_flat_channels` is a descriptive flag only.")
qc_df = pd.DataFrame(qc)
print(f"\nQC: {int(qc_df['passed'].sum())}/{len(qc_df)} passed")

## 12. Seizure-free recordings and the LOSO association question (NOT decided here)

The validated `assign_fold_context` (EEG_loso_evaluation_splitting.ipynb) assigns an
INTERICTAL epoch to the fold of `next_seizure_id`, else `previous_seizure_id`, **within its
own recording**, and asserts that one exists. Epochs from seizure-free recordings have
neither, so the validated LOSO code cannot place them. Choosing how is a methodological
decision not fixed by the repository or the supervisor requirements; the numbers below
quantify the situation and the options are reported, not implemented.

In [ ]:
lab5 = pd.Series(lab_rows[5], index=epoch_meta.index)
inter = epoch_meta[lab5 == "INTERICTAL"]
sf = inter["recording_is_seizure_free"]
print(f"INTERICTAL epochs (H=5, validated rule): {len(inter)} = {len(inter)*30/3600:.1f} h; "
      f"in seizure-free recordings: {int(sf.sum())} ({sf.mean():.1%}); in seizure recordings: {int((~sf).sum())}")
# zero-TEST-interictal folds under the validated within-recording association
pre5 = epoch_meta[lab5 == "PREICTAL"]
elig = pre5.groupby(["patient_id", "next_seizure_id"]).size().reset_index()[["patient_id", "next_seizure_id"]]
ctx = inter.loc[~sf, "next_seizure_id"].where(inter.loc[~sf, "next_seizure_id"].notna(), inter.loc[~sf, "previous_seizure_id"])
n_test_inter = ctx.value_counts()
elig["test_interictal_epochs_validated_rule"] = elig["next_seizure_id"].map(n_test_inter).fillna(0).astype(int)
print(f"eligible seizures (H=5): {len(elig)}; with 0 TEST interictal epochs under the validated rule: "
      f"{int((elig['test_interictal_epochs_validated_rule'] == 0).sum())}")
# how many seizure-free interictal epochs could be certified under the timeline rule
g_ok = (inter["timeline_available"] & (inter["global_time_to_next_seizure_sec"].fillna(np.inf) > 1800) &
        (inter["global_time_since_previous_seizure_sec"].fillna(np.inf) > 1800))
print(f"seizure-free-recording INTERICTAL epochs also >= 30 min from every seizure on the timeline: "
      f"{int((g_ok & sf).sum())}/{int(sf.sum())}; without a timeline (chb24 unsummarized files): "
      f"{int((sf & ~inter['timeline_available']).sum())}")
elig.to_csv(VAL / "eligible_seizures_test_interictal_validated_rule.csv", index=False)

**Options for associating seizure-free INTERICTAL epochs with LOSO folds** (to be decided
with the supervisor before the labeling/LOSO re-run):

1. **Timeline generalization of the validated rule** -- assign each interictal epoch to the
   fold of its next seizure on the patient timeline (`global_next_seizure_id`), else the
   previous one. Closest to the existing rule; fold sizes become very unequal (all interictal
   hours before a seizure go to that seizure's fold).
2. **Balanced chronological interictal blocks** -- split each patient's interictal time into
   as many contiguous blocks as there are eligible seizures and pair block *k* with seizure
   *k* (common in the seizure-prediction literature); needs a guard gap between blocks.
3. **Interictal as TRAIN-only** -- keep the validated fold rule for TEST and use seizure-free
   recordings only in TRAIN. Does not fix the zero-TEST-interictal folds.
4. **Separate interictal cross-validation** -- evaluate FA/hour over a leave-one-block-out
   of interictal data independently from the seizure LOSO.

A related, separate decision: whether labeling should use the patient-timeline distances
(`global_*`, "≥ 30 min from ANY seizure", across files) instead of the validated
within-recording distances; Section 10 reports how many labels differ. Unsummarized chb24
files have no clock times, so their distance to other files' seizures cannot be certified.

## 13. Save manifests, report, and re-open every output

In [ ]:
subj_note = {"chb21": "same subject as chb01 (recorded 1.5 years later)", "chb01": "same subject as chb21"}
pat_manifest = patients.merge(dataset_counts, on="patient_id", how="left")
pat_manifest["subject_note"] = pat_manifest["patient_id"].map(subj_note).fillna("")
pat_manifest.to_csv(OUT / "patient_inclusion_manifest.csv", index=False)

rec_manifest = recs.drop(columns=["declared_channels", "seizures"]).merge(
    rec_status[["recording_id", "status"]], on="recording_id").merge(
    prov[["recording_id", "duration_sec", "n_times", "n_epochs", "leftover_sec", "download_status"]], on="recording_id", how="left")
rec_manifest["seizures_json"] = recs["seizures"].values
rec_manifest.to_csv(OUT / "recording_manifest.csv", index=False)
rec_manifest.to_parquet(OUT / "recording_manifest.parquet", index=False)
excluded = rec_manifest[rec_manifest["status"] != "processed"][["patient_id", "edf_name", "recording_id", "n_seizures", "status", "exclusion_reason_pre"]]
if failures:
    fl = pd.DataFrame(failures).set_index("recording_id")["error"]
    excluded["failure_error"] = excluded["recording_id"].map(fl)
excluded.to_csv(OUT / "excluded_recordings.csv", index=False)

chan_def = pd.DataFrame({"channel_order": range(len(CHANNELS)), "channel": CHANNELS,
                         "channel_normalized": [cp.normalize_channel_name(c) for c in CHANNELS]})
chan_def.to_csv(OUT / "channel_set.csv", index=False)
chan_manifest.to_csv(OUT / "channel_montage_manifest.csv", index=False)
seizures_all.to_csv(OUT / "seizure_events.csv", index=False)
seizures_all.to_parquet(OUT / "seizure_events.parquet", index=False)
epoch_meta.to_parquet(OUT / "epoch_metadata.parquet", index=False)
prov.assign(channel_map=prov["channel_map"].apply(json.dumps), channels=prov["channels"].apply(json.dumps),
            legacy_channels=prov["legacy_channels"].apply(json.dumps), runtime_channels=prov["runtime_channels"].apply(json.dumps),
            duplicate_labels_identical=prov["duplicate_labels_identical"].apply(json.dumps)
            ).to_csv(OUT / "processing_provenance.csv", index=False)
dataset_counts.to_csv(OUT / "dataset_counts.csv", index=False)
qc_df.to_csv(OUT / "qc_checks.csv", index=False)

schema = {
    "key": KEY, "channels": CHANNELS, "channels_normalized": [cp.normalize_channel_name(c) for c in CHANNELS],
    "features": cp.FEATURES,
    "channelwise_columns_order": "channel-major, feature-minor: for ch in channels: for f in features -> f'{f}__{ch_norm}'",
    "channelwise_columns": CW_COLS,
    "averaged_columns": cp.FEATURES,
    "definitions": {
        "std/var/rms": "per channel over the 30 s epoch (channel-wise); over the flattened all-channel epoch (averaged)",
        "delta/theta/alpha/beta": "Welch PSD (nperseg=2 s, scipy defaults), trapezoid over inclusive bands 0.5-4/4-8/8-13/13-30 Hz; averaged = mean over channels",
        "rel_*": "band / (delta+theta+alpha+beta+1e-12); averaged uses the channel-averaged band powers",
        "theta_alpha_ratio / delta_beta_ratio": "theta/(alpha+1e-12), delta/(beta+1e-12); averaged uses averaged band powers",
        "entropy": "normalized Shannon entropy of Welch PSD over 1-40 Hz; averaged = mean over channels",
        "units": "signal in volts (MNE); powers in V^2",
    },
    "preprocessing": {"filter": "MNE FIR band-pass 0.5-40 Hz (MNE defaults, zero-phase)", "epoch_sec": 30.0, "overlap": 0.0,
                      "leftover": "dropped", "sfreq": 256.0, "artifact_rejection": "NOT IMPLEMENTED"},
}
(OUT / "feature_schema.json").write_text(json.dumps(schema, indent=1))
print("saved manifests / tables")

In [ ]:
# Re-open every saved output and verify it against the in-memory objects.
reopen = {}
em = pd.read_parquet(OUT / "epoch_metadata.parquet")
av = pd.read_parquet(av_path)
pf = pq.ParquetFile(cw_path)
reopen["epoch_metadata_shape"] = em.shape
reopen["channelwise_shape"] = (pf.metadata.num_rows, pf.metadata.num_columns)
reopen["averaged_shape"] = av.shape
reopen["cw_columns_ok"] = pf.schema_arrow.names == KEY + CW_COLS
reopen["av_columns_ok"] = av.columns.tolist() == KEY + cp.FEATURES
keys_meta = em[KEY].astype(str).to_numpy()
reopen["keys_aligned_meta_av"] = bool((keys_meta == av[KEY].astype(str).to_numpy()).all())
ok_keys = ok_fin = True
ok_avg = {f: True for f in ["delta", "theta", "alpha", "beta", "entropy"]}
pos = 0
for batch in pf.iter_batches(batch_size=20000):
    b = batch.to_pandas()
    n = len(b)
    ok_keys &= bool((b[KEY].astype(str).to_numpy() == keys_meta[pos:pos + n]).all())
    ok_fin &= bool(np.isfinite(b[CW_COLS].to_numpy()).all())
    # the averaged table must equal the mean over channels of the channel-wise band powers (same epochs)
    for f in ok_avg:
        cols = [f"{f}__{cp.normalize_channel_name(c)}" for c in CHANNELS]
        ok_avg[f] &= bool(np.allclose(b[cols].mean(axis=1).to_numpy(), av[f].to_numpy()[pos:pos + n], rtol=1e-12, atol=0))
    pos += n
reopen["keys_aligned_meta_cw"] = ok_keys and pos == len(em)
reopen["cw_finite"] = ok_fin
reopen["av_finite"] = bool(np.isfinite(av[cp.FEATURES].to_numpy()).all())
for f, v in ok_avg.items():
    reopen[f"avg_{f}_equals_mean_of_channelwise"] = v
reopen["seizure_events_rows"] = len(pd.read_parquet(OUT / "seizure_events.parquet"))
reopen["recording_manifest_rows"] = len(pd.read_parquet(OUT / "recording_manifest.parquet"))
reopen["schema_json_channelwise_matches"] = json.loads((OUT / "feature_schema.json").read_text())["channelwise_columns"] == CW_COLS
for f in ["patient_inclusion_manifest.csv", "excluded_recordings.csv", "channel_set.csv", "channel_montage_manifest.csv",
          "dataset_counts.csv", "qc_checks.csv", "processing_provenance.csv"]:
    reopen[f"rows_{f}"] = len(pd.read_csv(OUT / f))
print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in reopen.items()}, indent=1))
assert all(v for k, v in reopen.items() if isinstance(v, bool))

In [ ]:
PROTECTED_HASH_AFTER = {k: sha256(REPO / k) for k in PROTECTED_HASH_BEFORE}
changed = [k for k in PROTECTED_HASH_BEFORE if PROTECTED_HASH_BEFORE[k] != PROTECTED_HASH_AFTER[k]]
print(f"protected legacy outputs unchanged: {len(changed) == 0} ({len(PROTECTED_HASH_BEFORE)} files); changed: {changed}")
qc_df = pd.concat([qc_df, pd.DataFrame([{"check": "validated legacy outputs unchanged", "passed": len(changed) == 0,
                                         "detail": f"{len(PROTECTED_HASH_BEFORE)} files sha256-compared"}])], ignore_index=True)
qc_df.to_csv(OUT / "qc_checks.csv", index=False)

out_files = sorted(p for p in OUT.glob("*.*"))
report = {
    "dataset_version": "chbmit_multipatient_v1",
    "created_utc": pd.Timestamp.now('UTC').isoformat(),
    "full_run": not DEV_PATIENTS,
    "versions": VERSIONS,
    "raw_dir": str(RAW_DIR),
    "channel_rule": {k: v for k, v in rule.items()},
    "inclusion_rule": {"min_eligible_seizures": cp.MIN_ELIGIBLE_SEIZURES, "min_interictal_hours": cp.MIN_INTERICTAL_HOURS},
    "n_cases_total": len(patients), "n_cases_included": int(patients["included"].sum()),
    "n_recordings_processed": len(prov), "n_processing_failures": len(failures),
    "n_epochs": len(epoch_meta), "hours_processed": float(prov["duration_sec"].sum() / 3600),
    "channelwise_dims": [len(epoch_meta), len(CW_COLS)], "averaged_dims": [len(epoch_meta), len(cp.FEATURES)],
    "backward_validation": bv,
    "qc": qc_df.to_dict(orient="records"),
    "reopen_checks": {k: (list(v) if isinstance(v, tuple) else v) for k, v in reopen.items()},
    "artifact_rejection": "NOT IMPLEMENTED (descriptive n_flat_channels flag only)",
    "unresolved_decisions": ["seizure-free INTERICTAL -> LOSO fold association (Section 12 options)",
                             "within-recording vs patient-timeline distances for labeling (Section 10)"],
    "output_sha256": {p.name: sha256(p) for p in out_files},
}
(OUT / "preprocessing_validation_report.json").write_text(json.dumps(report, indent=1, default=str))
lines = ["# CHB-MIT multipatient dataset v1 -- preprocessing validation report", "",
         f"- full run: {report['full_run']}", f"- cases included: {report['n_cases_included']}/{report['n_cases_total']}",
         f"- recordings processed: {report['n_recordings_processed']} (failures: {report['n_processing_failures']})",
         f"- hours: {report['hours_processed']:.1f}; epochs: {report['n_epochs']}",
         f"- channel set ({len(CHANNELS)}): {', '.join(CHANNELS)}",
         f"- channel-wise dims: {report['channelwise_dims']}; averaged dims: {report['averaged_dims']}",
         f"- backward validation passed: {bv['all_checks_passed']} (max rel feature diff {bv['features_legacy23_max_rel_diff']:.2e})",
         f"- QC: {int(qc_df['passed'].sum())}/{len(qc_df)} passed", "- artifact rejection: NOT IMPLEMENTED", "",
         "## QC checks", ""] + [f"- [{'PASS' if r.passed else 'FAIL'}] {r.check}: {r.detail}" for r in qc_df.itertuples()] + \
        ["", "## Unresolved decisions", ""] + [f"- {u}" for u in report["unresolved_decisions"]]
(OUT / "preprocessing_validation_report.md").write_text("\n".join(lines))
print("\n".join(lines))